## CNN regression

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Check available hardware
import torch
if torch.cuda.is_available():
    print(f'GPU is available, using GPU {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
elif torch.mps.is_available():
    print(f'Apple Metal Performance Shaders framework is available, using {torch.device("mps")}')
    device = torch.device('mps')
else:
    print(f'Hardware acceleration not available, using CPU')
    device = torch.device('cpu')

## Dataset: UTK Face

See here: https://susanqq.github.io/UTKFace/

In [ ]:
import os
from PIL import Image

path = os.path.expanduser('~/data10/utk-face/UTKface_inthewild')
image_files = [f for f in os.listdir(path) if f.endswith('.jpg')]

img = Image.open(os.path.join(path, image_files[0]))
img

In [ ]:
from torch.utils.data import Dataset
import os
class UTKFaceDataset(Dataset):
    """ Custom dataset class to load in UTK Face dataset elements
        Files have the name format [age]_[gender]_[race]_[date&time].jpg
    """
    def __init__(self,
                 root=os.path.expanduser('~/data10/utk-face/UTKface_inthewild'), 
                 transform=None):

    def __len__(self):

    def __getitem__(self, idx):


In [ ]:
dataset = UTKFaceDataset()

fig, ax = plt.subplots(1, 4)
for ii in range(4):
    idx = np.random.choice(len(dataset))
    img, age = dataset[idx]
    ax[ii].imshow(img)
    ax[ii].set_title(f'Age = {age:.1f}')
    ax[ii].axis('off')

In [ ]:
from torchvision.transforms import v2

transform = v2.Compose([
    # Transforms go here
])
dataset = UTKFaceDataset(transform=transform)

fig, ax = plt.subplots(1, 4)
for ii in range(4):
    idx = np.random.choice(len(dataset))
    img, age = dataset[idx]
    img = denormalize(img)
    ax[ii].imshow(img.permute(1, 2, 0))
    ax[ii].set_title(f'Age = {age:.0f} yrs')
    ax[ii].axis('off')

In [ ]:
from torch.utils.data import random_split
train_dataset, val_dataset, test_dataset = random_split(dataset, lengths=[0.6, 0.2, 0.2])
print(len(train_dataset), len(val_dataset), len(test_dataset))

## CNN implementation

This is a harder problem so we're going to build a bigger neural network. Two tools will come in handy:

- Residual (or skip) connections
- Batch normalization

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, c_in, c_out, **conv_kwargs):
        super().__init__()        

    def forward(self, x):

class CNN_Regression(nn.Module):
    def __init__(self):
        super().__init__()

        # Let's do a list of stages, each containing some CNN layers
        # After each stage, we'll do a MaxPool and downsample
        self.stages = nn.ModuleList()

        # Input data is [B, 3, 128, 128]
        # First stage projects data onto a large feature map
        
        # First stage output is: 
        
        # Let's just stack a bunch of Residual blocks in each stage
        for ii in range(4):

        # How many feature maps do I have?
        # How big is each feature map?

        # I have a lot of options in what I do here
        # Global average/max pooling, big linear layer, 
        # keep throwing CNN stages until it's size 1?

        # Add a classifier head

    def forward(self, x):
        # x has shape [B, 3, H, W]
        # Iterate through my nn.ModuleList of stages

        # Apply my classifier head
        return x

In [ ]:
model = CNN_Regression().to(device)

# Ensure we can make a prediction with this model, no issues
img, age = dataset[0]
pred = model(img.to(device)[None])
print(f'Model makes prediction which is {pred}')

# How many weights are there now?
num_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CNN has {num_trainable_params} trainable parameters')

model

## Model training

In [ ]:
from tqdm.notebook import tqdm

def train(model, optimizer, loss_func, train_dataset, val_dataset, batch_size=32, num_epochs=10, num_workers=4, scheduler=None):
    model = model.to(device)

    # Build training and validation loaders
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=True, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False, pin_memory=True)

    train_loss, val_loss = [], []
    train_mad, val_mad = [], [] # Tracking mean absolute deviation in units of years

    for epoch in range(num_epochs):
        model.train()
        train_loss_ = 0.
        train_mad_ = 0.
        for xb, yb in tqdm(train_loader, leave=False):
            # Send data to device

            # Make a prediction, compute the loss

            # Backpropagate, apply gradients, reset for next pass

            # Log loss and metrics
            train_loss_ += loss.item() * yb.shape[0]
            train_mad_ += F.l1_loss(y_pred, yb, reduction='sum').item()
            
        train_loss.append(train_loss_ / len(train_loader.dataset))
        train_mad.append(train_mad_ / len(train_loader.dataset))

        model.eval()
        with torch.no_grad():
            val_loss_ = 0.
            val_mad_ = 0.
            for xb, yb in tqdm(val_loader, leave=False):
                # Send data to device

                # Make a prediction, compute the loss

                # Log loss and metrics
                val_loss_ += loss.item() * yb.shape[0]
                val_mad_ += F.l1_loss(y_pred, yb, reduction='sum').item()

            val_loss.append(val_loss_ / len(val_loader.dataset))
            val_mad.append(val_mad_ / len(val_loader.dataset))

        print(f'Epoch {epoch+1} | '
              f'Train loss { train_loss[-1]:.3f} | '
              f'Train MAD { train_mad[-1]:.3f} years | '
              f'Val loss { val_loss[-1]:.3f} | '
              f'Val MAD { val_mad[-1]:.3f} years '
        )

        # Consider early stopping
        # Consider learning rate scheduling

    return train_loss, train_mad, val_loss, val_mad

In [ ]:
model = CNN_Regression()
loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss, train_mad, val_loss, val_mad = train(
    model, optimizer, loss, train_dataset, val_dataset,
    batch_size=32, num_epochs=1, num_workers=8)

Let's run this script in the background while we move onto the next part

In [ ]:
results = torch.load(os.path.expanduser('~/scr10/cnn_regression_results.pt')
train_loss = results['train_loss']
train_mad = results['train_mad']
val_loss = results['val_loss']
val_mad = results['val_mad']

In [ ]:
fig, ax = plt.subplots(1, 2, sharex=True, figsize=(5, 2.5))

ax[0].plot(train_loss, color='black', label='Train')
ax[0].plot(val_loss, color='red', label='Validation')
ax[0].set(xlabel='Epoch', ylabel='Loss')

ax[1].plot(train_mad, color='black', label='Train')
ax[1].plot(val_mad, color='red', label='Validation')
ax[1].set(xlabel='Epoch', ylabel='Mean Absolute Deviation [yr]')

fig.legend(*ax[1].get_legend_handles_labels(), 
           loc='center left', bbox_to_anchor=[1, 0.5], framealpha=0)

plt.tight_layout()

## Evaluate on testing dataset

In [ ]:
# Load the model back in
model = CNN_Regression()
model.load_state_dict(results['model_weights'])
model = model.to(device)

# Make sure we use the correct test dataset
test_dataset = torch.utils.data.Subset(dataset, results['test_indices'])
print(f'Test dataset has length {len(test_dataset)}')

In [ ]:
y_true, y_pred = [], []
model.eval()

test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=8)
with torch.no_grad():
    for xb, yb in tqdm(test_loader):
        yp = model(xb.to(device))
        
        y_true.append(yb.numpy())
        y_pred.append(yp.squeeze().cpu().numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
print(y_true.shape, y_pred.shape)

In [ ]:
from sklearn.metrics import r2_score
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(y_true, y_pred, s=5, alpha=0.5, label=f'Test ($R^2 = {r2_score(y_true, y_pred):.2f}$)')
ax.plot([0,100], [0, 100], color='red', linestyle='--')
ax.legend();

print(f'Mean absolute difference: {np.mean(np.abs(y_true - y_pred)):.1f} yrs')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ii, ax in enumerate(axes.flatten()):
    x, y = test_dataset[np.random.choice(len(test_dataset))]
    with torch.no_grad():
        yp = model(x.to(device)[None]).squeeze().item()
    ax.imshow(denormalize(x.cpu()).permute(1, 2, 0)) #Denormalize images
    ax.set_title(f'True age: {y:.0f} yrs\nPred age: {yp:.0f} yrs')
    ax.axis('off')

plt.tight_layout()